# Suite P3PRT — Printing Python 3 modules from the store

`Programs.Python3.Print` prints a module as the store holds it: a reference object that
`Programs.Framework.TheWorkspace` owns by its path, holding its tree as one value, its `body`. Each statement is a
value object, a branch of the union `Programs.Python3.Statement` tagged by its kind in lower case, holding the
statements of its blocks and its clauses as its properties (`body`, `orelse`, `finalbody`, `handlers`, `cases`), in
their containers' order; its expressions are mbse-expressions' terms, which `Python3.Text.ToText` writes. The layout
is the 0.3 printer's: four spaces a level, `elif` for an `else` that is one `if`, PEP 8's blank lines around
definitions, comments kept with the definition they lead, trailing comments on the line they end. Where a statement's
place takes a bare tuple, `yield` or named expression, it is written bare.

Each case builds its module in a transaction of its own, its tree written in the builders' own syntax, the module's
`body` a list of the union's branches (`lambda u: getattr(u, "if")(...)` where a tag is a Python keyword), its
expressions read from Python source by `Python3.Text.FromText`, and checks the printed text, which CPython's `ast`
must parse. The case setup is below.

In [ ]:
import ast

from mbse.Expressions import Framework
from mbse.Expressions.Framework import Objects, Terms
from mbse.Expressions.IEEE754 import Expressions as IEEE754
from mbse.Expressions.ISOIEC11404 import Expressions as ISOIEC11404
from mbse.Expressions.Python import Patterns as PythonPatterns, Python3, Types as PythonTypes
from mbse.Expressions.Python.Python3.Text import FromText as X
from mbse.Programs import Python3 as Py
from mbse.Programs.Framework import Model
from mbse.Programs.Python3 import Print, Statements
from mbse.Schemas.Framework import Builders as B
from support import raises

INSTALLS = (Framework.install, ISOIEC11404.install, IEEE754.install, Python3.install, PythonTypes.install,
            PythonPatterns.install, Model.install, Py.install)
store = B.Store()
for install in INSTALLS:
    store.transaction(install)
R = store.registered


def build(path, body):
    """The module at `path`, built in a transaction of its own: `body` its statements' writers, the builders'."""
    store.transaction(lambda t: t.uid(path).mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"ws": R("Programs.Framework.Workspace"), "m": R("Programs.Python3.Module")})
        .spec([lambda _: _.ws.files({path: lambda e: e.file(_.m)}), lambda _: _.m.body(body)])]))]))


def printed(path, body):
    """The text of the module built at `path`, which CPython's `ast` parses."""
    build(path, body)
    text = Print.print_module(store, path)
    ast.parse(text)
    return text

<a id="P3PRT-01"></a>
## P3PRT-01 · Every simple statement kind prints alone, on its line

An expression statement, assignments plain, augmented and annotated, `assert`, `pass`, `break`, `continue`, `del`,
`return`, `raise`, imports with their aliases, relative and `*`, `global`, `nonlocal`, a type alias with its type
parameters, and a comment, whose text follows its `#`. `lazy` imports (Python 3.15) print too.

In [ ]:
text = printed("simple.py", [
    lambda u: u.expr(lambda s: s.value(X("f(x)"))),
    lambda u: u.assign(lambda s: s.targets([X("x")]).value(X("1"))),
    lambda u: u.augassign(lambda s: s.target(X("x")).op("<<").value(X("2"))),
    lambda u: u.annassign(lambda s: s.target(X("x")).annotation(X("int")).value(X("1"))),
    lambda u: u.annassign(lambda s: s.target(X("self.y")).annotation(X("list[int]"))),
    lambda u: getattr(u, "assert")(lambda s: s.test(X("x > 0"))),
    lambda u: getattr(u, "assert")(lambda s: s.test(X("x")).msg(X("'m'"))),
    lambda u: getattr(u, "pass")(lambda s: s),
    lambda u: getattr(u, "break")(lambda s: s),
    lambda u: getattr(u, "continue")(lambda s: s),
    lambda u: u.delete(lambda s: s.targets([X("a"), X("b[0]"), X("c.d")])),
    lambda u: getattr(u, "return")(lambda s: s),
    lambda u: getattr(u, "return")(lambda s: s.value(X("x"))),
    lambda u: getattr(u, "raise")(lambda s: s),
    lambda u: getattr(u, "raise")(lambda s: s.exc(X("E"))),
    lambda u: getattr(u, "raise")(lambda s: s.exc(X("E('m')")).cause(X("error"))),
    lambda u: getattr(u, "import")(lambda s: s.names([lambda a: a.name("os"),
                                                      lambda a: a.name("numpy.linalg").asname("la")])),
    lambda u: u.importfrom(lambda s: s.level(1).names([lambda a: a.name("a")])),
    lambda u: u.importfrom(lambda s: s.level(2).module("pkg.sub").names([lambda a: a.name("b").asname("c"),
                                                                         lambda a: a.name("d")])),
    lambda u: u.importfrom(lambda s: s.level(0).module("os").names([lambda a: a.name("*")])),
    lambda u: getattr(u, "global")(lambda s: s.names(["a", "b"])),
    lambda u: getattr(u, "nonlocal")(lambda s: s.names(["c"])),
    lambda u: u.typealias(lambda s: s.name(X("Pair")).type_params([lambda t: t.typevar(lambda v: v.name("T"))])
                          .value(X("tuple[T, T]"))),
    lambda u: u.typealias(lambda s: s.name(X("Alias")).value(X("int | None"))),
    lambda u: u.comment(lambda s: s.text(" a note")),
])
assert text == """\
f(x)
x = 1
x <<= 2
x: int = 1
self.y: list[int]
assert x > 0
assert x, 'm'
pass
break
continue
del a, b[0], c.d
return
return x
raise
raise E
raise E('m') from error
import os, numpy.linalg as la
from . import a
from ..pkg.sub import b as c, d
from os import *
global a, b
nonlocal c
type Pair[T] = tuple[T, T]
type Alias = int | None
# a note
"""
build("lazy.py", [  # Python 3.15's, which CPython's `ast` before it does not parse
    lambda u: getattr(u, "import")(lambda s: s.is_lazy(True).names([lambda a: a.name("json")])),
    lambda u: u.importfrom(lambda s: s.is_lazy(True).level(0).module("os").names([lambda a: a.name("path")])),
])
assert Print.print_module(store, "lazy.py") == "lazy import json\nlazy from os import path\n"

<a id="P3PRT-02"></a>
## P3PRT-02 · Every compound statement kind prints, with its clauses and its parts

`if`, `while` and `for` with their `else`, `async for` and `async with`, `try` with its `except` clauses (a type, a
tuple, a name, none), `else` and `finally`, `try` with `except*`, `with` and its items, `match` with its cases'
patterns and guards, and function and class definitions with their decorators, type parameters, parameters of every
kind, return type, bases and keywords. Each holds its blocks and its clauses as its own properties, nested in the
module's value.

In [ ]:
loops = printed("loops.py", [
    lambda u: getattr(u, "if")(lambda s: s.test(X("a"))
                               .body([lambda b: getattr(b, "pass")(lambda p: p)])
                               .orelse([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: getattr(u, "while")(lambda s: s.test(X("a < b"))
                                  .body([lambda b: getattr(b, "pass")(lambda p: p)])
                                  .orelse([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: getattr(u, "for")(lambda s: s.target(X("x")).iter(X("range(3)"))
                                .body([lambda b: getattr(b, "pass")(lambda p: p)])
                                .orelse([lambda b: getattr(b, "pass")(lambda p: p)])),
])
assert loops == """\
if a:
    pass
else:
    pass
while a < b:
    pass
else:
    pass
for x in range(3):
    pass
else:
    pass
"""
asynchronous = printed("async.py", [
    lambda u: u.asyncfunctiondef(lambda f: f.name("main").body([
        lambda b: b.asyncfor(lambda s: s.target(X("x")).iter(X("stream()")).body([
            lambda b: b.asyncwith(lambda w: w.items([lambda i: i.context_expr(X("lock"))]).body([
                lambda b: b.expr(lambda e: e.value(X("await handle(x)")))]))]))])),
])
assert asynchronous == """\
async def main():
    async for x in stream():
        async with lock:
            await handle(x)
"""
tries = printed("try.py", [
    lambda u: getattr(u, "try")(lambda s: s
        .body([lambda b: getattr(b, "pass")(lambda p: p)])
        .handlers([
            lambda h: h.excepthandler(lambda e: e.type(X("ValueError")).name("e")
                                      .body([lambda b: getattr(b, "pass")(lambda p: p)])),
            lambda h: h.excepthandler(lambda e: e.type(X("(KeyError, IndexError)"))
                                      .body([lambda b: getattr(b, "pass")(lambda p: p)])),
            lambda h: h.excepthandler(lambda e: e.body([lambda b: getattr(b, "pass")(lambda p: p)]))])
        .orelse([lambda b: getattr(b, "pass")(lambda p: p)])
        .finalbody([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: u.trystar(lambda s: s
        .body([lambda b: getattr(b, "pass")(lambda p: p)])
        .handlers([lambda h: h.excepthandler(lambda e: e.type(X("OSError")).name("group")
                                             .body([lambda b: getattr(b, "pass")(lambda p: p)]))])),
])
assert tries == """\
try:
    pass
except ValueError as e:
    pass
except (KeyError, IndexError):
    pass
except:
    pass
else:
    pass
finally:
    pass
try:
    pass
except* OSError as group:
    pass
"""
withs = printed("with.py", [
    lambda u: getattr(u, "with")(lambda s: s
        .items([lambda w: w.context_expr(X("open(path)")).optional_vars(X("f")),
                lambda w: w.context_expr(X("a")).optional_vars(X("(b, c)")),
                lambda w: w.context_expr(X("lock"))])
        .body([lambda b: getattr(b, "pass")(lambda p: p)])),
])
assert withs == "with open(path) as f, a as (b, c), lock:\n    pass\n"

In [ ]:
def value(term):
    return lambda u: u.matchvalue(lambda v: v.value(term))


def capture(name):
    return lambda u: u.matchas(lambda m: m.name(name))


match = printed("match.py", [
    lambda u: u.match(lambda s: s.subject(X("command")).cases([
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchsequence(lambda s: s.delimiters("[]").patterns([capture("action")])))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchclass(lambda c: c.cls(X("Point")).patterns([value(X("0"))])
                                            .kwd_attrs(["y"]).kwd_patterns([capture("y")]))).guard(X("y > 0"))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchmapping(lambda m: m.keys([X("'k'")]).patterns([capture("v")]).rest("rest")))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchas(lambda m: m.name("n").pattern(lambda p: p.matchor(
                lambda o: o.patterns([value(X("1")), value(X("2"))])))))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchsequence(lambda s: s.patterns([capture("a"), lambda p: p.matchstar(
                lambda m: m.name("others"))])))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchsingleton(lambda m: m.value("None")))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchsequence(lambda s: s.delimiters("()").patterns([capture("x")])))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda c: c.matchcase(lambda m: m
            .pattern(lambda u: u.matchas(lambda m: m))
            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
    ])),
])
assert match == """\
match command:
    case [action]:
        pass
    case Point(0, y=y) if y > 0:
        pass
    case {'k': v, **rest}:
        pass
    case 1 | 2 as n:
        pass
    case a, *others:
        pass
    case None:
        pass
    case (x,):
        pass
    case _:
        pass
"""

In [ ]:
definitions = printed("definitions.py", [
    lambda u: u.functiondef(lambda f: f.decorator_list([X("cache"), X("route('/a')")]).name("f").type_params([
        lambda t: t.typevar(lambda v: v.name("T").bound(X("int")).default_value(X("bool"))),
        lambda t: t.typevartuple(lambda v: v.name("Ts")),
        lambda t: t.paramspec(lambda v: v.name("P"))]).args(lambda a: a
        .posonlyargs([lambda g: g.arg("a")])
        .args([lambda g: g.arg("b").annotation(X("T")), lambda g: g.arg("c").default_value(X("1")),
               lambda g: g.arg("d").annotation(X("int")).default_value(X("2"))])
        .vararg(lambda g: g.arg("rest").annotation(X("*Ts")))
        .kwonlyargs([lambda g: g.arg("e"), lambda g: g.arg("k").default_value(X("None"))])
        .kwarg(lambda g: g.arg("options"))).returns(X("T | None"))
        .body([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: u.classdef(lambda c: c.decorator_list([X("dataclass")]).name("C")
        .type_params([lambda t: t.typevar(lambda v: v.name("T"))])
        .bases([X("Base[T]"), X("Mixin")]).keywords([lambda k: k.arg("metaclass").value(X("Meta")),
                                                     lambda k: k.value(X("options"))])
        .body([lambda b: getattr(b, "pass")(lambda p: p)])),
])
assert definitions == """\
@cache
@route('/a')
def f[T: int = bool, *Ts, **P](a, /, b: T, c=1, d: int = 2, *rest: *Ts, e, k=None, **options) -> T | None:
    pass


@dataclass
class C[T](Base[T], Mixin, metaclass=Meta, **options):
    pass
"""
bare = printed("bare.py", [
    lambda u: u.functiondef(lambda f: f.name("f").body([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: u.functiondef(lambda f: f.name("g").args(lambda a: a.kwonlyargs([lambda g: g.arg("k")]))
                            .body([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: u.classdef(lambda c: c.name("C").body([lambda b: getattr(b, "pass")(lambda p: p)])),
])
assert bare == "def f():\n    pass\n\n\ndef g(*, k):\n    pass\n\n\nclass C:\n    pass\n"

<a id="P3PRT-03"></a>
## P3PRT-03 · Layout: blank lines around definitions, and comments

Two blank lines around a top-level definition and one around a nested one; comments directly before a definition stay
with it, the blank lines before them; a trailing comment ends the line of what it follows, and the first of a body
ends its header's line.

In [ ]:
layout = printed("layout.py", [
    lambda u: getattr(u, "import")(lambda s: s.names([lambda a: a.name("os")])),
    lambda u: u.comment(lambda s: s.text(" lead")),
    lambda u: u.functiondef(lambda f: f.name("f").body([
        lambda b: b.comment(lambda s: s.text(" on the header").trailing(True)),
        lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: u.classdef(lambda c: c.decorator_list([X("d")]).name("C").body([
        lambda b: b.expr(lambda s: s.value(X("'doc'"))),
        lambda b: b.functiondef(lambda f: f.name("m").args(lambda a: a.args([lambda g: g.arg("self")]))
                                .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda b: b.comment(lambda s: s.text(" first")),
        lambda b: b.comment(lambda s: s.text(" second")),
        lambda b: b.functiondef(lambda f: f.name("n").args(lambda a: a.args([lambda g: g.arg("self")]))
                                .body([lambda b: getattr(b, "pass")(lambda p: p)])),
        lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: u.comment(lambda s: s.text(" trailing").trailing(True)),
    lambda u: u.assign(lambda s: s.targets([X("x")]).value(X("1"))),
])
assert layout == """\
import os


# lead
def f():  # on the header
    pass


@d
class C:
    'doc'

    def m(self):
        pass

    # first
    # second
    def n(self):
        pass

    pass  # trailing


x = 1
"""

<a id="P3PRT-04"></a>
## P3PRT-04 · `elif` for an `else` that is one `if`

An `if` whose `orelse` is one `if` prints as `elif`, in a chain; an `orelse` holding anything else beside it, a comment
among them, prints as `else` with the `if` nested.

In [ ]:
chain = printed("chain.py", [
    lambda u: getattr(u, "if")(lambda i: i.test(X("a")).body([lambda b: getattr(b, "pass")(lambda p: p)]).orelse([
        lambda o: getattr(o, "if")(lambda j: j.test(X("b")).body([
            lambda b: b.comment(lambda s: s.text(" c").trailing(True)),
            lambda b: getattr(b, "pass")(lambda p: p)]).orelse([
            lambda o: getattr(o, "if")(lambda k: k.test(X("c"))
                                       .body([lambda b: getattr(b, "pass")(lambda p: p)])
                                       .orelse([lambda b: getattr(b, "pass")(lambda p: p)]))]))])),
])
assert chain == "if a:\n    pass\nelif b:  # c\n    pass\nelif c:\n    pass\nelse:\n    pass\n"
nested = printed("nested.py", [
    lambda u: getattr(u, "if")(lambda i: i.test(X("a")).body([lambda b: getattr(b, "pass")(lambda p: p)]).orelse([
        lambda o: o.comment(lambda s: s.text(" keeps else")),
        lambda o: getattr(o, "if")(lambda j: j.test(X("b")).body([lambda b: getattr(b, "pass")(lambda p: p)]))])),
])
assert nested == "if a:\n    pass\nelse:\n    # keeps else\n    if b:\n        pass\n"

<a id="P3PRT-05"></a>
## P3PRT-05 · Precedence where a statement's place takes more than an expression

`ToText` writes a tuple in parentheses and a named expression or a `yield` in their own; where a statement's place
takes them bare, they print bare: a tuple as an expression statement, an assignment's targets and value, a `return`'s
value, a `for`'s target and iterable and a `match`'s subject; a `yield` as an expression statement and an assignment's
value; a named expression as the test of an `if`, an `elif` and a `while`, a guard, a decorator and a subject. Elsewhere
they keep their parentheses, and an `except` clause's tuple keeps its own.

In [ ]:
statements = printed("precedence.py", [
    lambda u: u.expr(lambda s: s.value(X("a, b"))),
    lambda u: u.expr(lambda s: s.value(X("yield a"))),
    lambda u: u.expr(lambda s: s.value(X("(x := 1)"))),
    lambda u: u.expr(lambda s: s.value(X("(a + b) * c"))),
    lambda u: u.expr(lambda s: s.value(X("()"))),
    lambda u: u.assign(lambda s: s.targets([X("a, b")]).value(X("b, a"))),
    lambda u: u.assign(lambda s: s.targets([X("x"), X("(y,)")]).value(X("yield"))),
    lambda u: u.assign(lambda s: s.targets([X("y")]).value(X("(x := 1)"))),
    lambda u: u.assign(lambda s: s.targets([X("f")]).value(X("lambda x: x"))),
    lambda u: u.assign(lambda s: s.targets([X("z")]).value(X("yield from g()"))),
    lambda u: u.augassign(lambda s: s.target(X("total")).op("+").value(X("yield"))),
    lambda u: u.annassign(lambda s: s.target(X("pair")).annotation(X("tuple[int, int]")).value(X("1, 2"))),
    lambda u: u.functiondef(lambda f: f.decorator_list([X("(d := make())")]).name("f").body([
        lambda b: getattr(b, "return")(lambda s: s.value(X("a, b"))),
        lambda b: getattr(b, "return")(lambda s: s.value(X("(x := 1)"))),
        lambda b: getattr(b, "return")(lambda s: s.value(X("(yield)")))])),
    lambda u: getattr(u, "for")(lambda s: s.target(X("k, v")).iter(X("a, b"))
                                .body([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: getattr(u, "if")(lambda i: i.test(X("(n := f())")).body([lambda b: getattr(b, "pass")(lambda p: p)])
                               .orelse([lambda o: getattr(o, "if")(lambda j: j.test(X("(m := g()) > 0")).body([
                                   lambda b: getattr(b, "pass")(lambda p: p)]))])),
    lambda u: getattr(u, "while")(lambda s: s.test(X("(line := read())"))
                                  .body([lambda b: getattr(b, "pass")(lambda p: p)])),
    lambda u: u.match(lambda s: s.subject(X("a, (b := c)")).cases([
        lambda c: c.matchcase(lambda m: m.pattern(lambda p: p.matchas(lambda a: a.name("x"))).guard(X("(y := x)"))
                              .body([lambda b: getattr(b, "pass")(lambda p: p)]))])),
    lambda u: getattr(u, "try")(lambda s: s.body([lambda b: getattr(b, "pass")(lambda p: p)]).handlers([
        lambda h: h.excepthandler(lambda e: e.type(X("(A, B)")).body([lambda b: getattr(b, "pass")(lambda p: p)]))])),
    lambda u: getattr(u, "assert")(lambda s: s.test(X("(a, b)")).msg(X("(x := 1)"))),
    lambda u: u.delete(lambda s: s.targets([X("(a, b)"), X("c")])),
    lambda u: getattr(u, "with")(lambda s: s.items([lambda w: w.context_expr(X("(yield)")).optional_vars(X("v"))])
                                 .body([lambda b: getattr(b, "pass")(lambda p: p)])),
])
assert statements == """\
a, b
yield a
(x := 1)
(a + b) * c
()
a, b = b, a
x = y, = yield
y = (x := 1)
f = lambda x: x
z = yield from g()
total += yield
pair: tuple[int, int] = 1, 2


@d := make()
def f():
    return a, b
    return (x := 1)
    return (yield)


for k, v in a, b:
    pass
if n := f():
    pass
elif (m := g()) > 0:
    pass
while line := read():
    pass
match a, (b := c):
    case x if y := x:
        pass
try:
    pass
except (A, B):
    pass
assert (a, b), (x := 1)
del (a, b), c
with (yield) as v:
    pass
"""

<a id="P3PRT-06"></a>
## P3PRT-06 · Reading the store: transactions, visitors, order, and every kind

`print_module` reads within the transaction open on the store, or within one of its own, and refuses a path the
workspace holds no file at; `print_file` prints a module's visitor and `print_statement` a statement's record, the one
`Terms.kind_of` reads of an element of the container that holds it, each within the transaction that read it.
Statements print in their container's order, the order of its index, whatever the order their elements were written
in. A clause is a value that knows no statement holding it: an `except` clause printed within its `try` with `*` is
`except*`, and printed alone, `except`. The printer prints every statement kind.

In [ ]:
def held(prop):
    """What a property holds, as `Objects.held` reads it: a container's `Listed`, whose `values` are in its order."""
    found = []
    prop.as_value(lambda a: found.append(Objects.held(a)) or a)
    return found[0]


build("order.py", {  # by index, written out of order
    1: lambda u: u.assign(lambda s: s.targets([X("y")]).value(X("2"))),
    2: lambda u: u.trystar(lambda s: s
        .handlers([lambda h: h.excepthandler(lambda e: e.type(X("E"))
                                             .body([lambda b: getattr(b, "pass")(lambda p: p)]))])
        .body([lambda b: getattr(b, "pass")(lambda p: p)])),
    0: lambda u: u.assign(lambda s: s.targets([X("x")]).value(X("1"))),
})
expected = "x = 1\ny = 2\ntry:\n    pass\nexcept* E:\n    pass\n"
assert Print.print_module(store, "order.py") == expected  # in a transaction of its own
with store.transaction():
    with raises(ValueError, match="transactions do not nest"):  # print_module opens its own: called with none open
        Print.print_module(store, "order.py")
    module = store.singleton(Model.WORKSPACE).files({"path": "order.py"})
    assert Print.print_file(module) == expected
    statements = [Terms.kind_of(element) for element in held(module.body).values]
    assert [tag for tag, _ in statements] == ["assign", "assign", "trystar"]
    assert [Print.print_statement(record) for _, record in statements] == [
        "x = 1", "y = 2", "try:\n    pass\nexcept* E:\n    pass"]
    ((tag, handler),) = [Terms.kind_of(element) for element in held(statements[2][1].handlers).values]
    assert tag == "excepthandler"
    assert Print.print_statement(handler) == "except E:\n    pass"  # alone, it knows no `try` with `*`
    with raises(TypeError, match="not a Programs.Python3.Module"):
        Print.print_file(statements[0][1])
    with raises(TypeError, match="no printer for the statement kind 'Programs.Python3.Module'"):
        Print.print_statement(module)
with raises(LookupError, match="'nope.py'"):
    Print.print_module(store, "nope.py")
build("empty.py", [])
assert Print.print_module(store, "empty.py") == ""
assert sorted(Print.KINDS) == sorted((*Statements.KINDS, "Error"))  # and invalid syntax, Programs.Python3.Error

<a id="P3PRT-07"></a>
## P3PRT-07 · Invalid syntax prints as written: a parsed file of good and bad statements prints back

An `Error` prints its text, its first line at its level and the others as the source indented them, and not its
`body`, the statements recovered within it, which its text holds; an `error` term its text and a `missing` term
nothing, as `ToText` writes them; an `Error` with no text, nothing. So each invalid part reads back as written, and the
rest in the printer's layout.

In [ ]:
from mbse.Programs.Python3 import Parse


def reprinted(text, path):
    """The text of the module `text` parses to, printed back."""
    store.transaction(Parse.parse(text, path))
    return Print.print_module(store, path)


# A file of good and bad statements: each `Error` and `error` term as written, the rest in the printer's layout
source = ("import os\n"
          "x = = 1\n"
          "def f(a):\n"
          "    y = (a + )\n"
          "    print 'old'\n"
          "    for in a:\n"
          "        pass\n"
          "    return a <> 2\n"
          "class C(:\n"
          "    z = 1\n"
          "w = f(1,,2)\n")
assert reprinted(source, "invalid_mixed.py") == ("import os\n"
                                         "x = = 1\n"
                                         "\n\n"
                                         "def f(a):\n"
                                         "    y = (a + )\n"
                                         "    print 'old'\n"
                                         "    for in a:\n"  # a `missing` target prints as nothing, its space with it
                                         "        pass\n"
                                         "    return a <> 2\n"
                                         "\n\n"
                                         "class C(:\n"
                                         "    z = 1\n"
                                         "w = f(1,,2)\n")
assert reprinted(reprinted(source, "invalid_mixed.py"), "invalid_again.py") == reprinted(source, "invalid_mixed.py")  # a fixed point
# An `Error`'s first line at its level, the others as the source indented them; its children are in its text
assert reprinted("if a:\n    try:\n        pass\n    except E +:\n        pass\n", "invalid_nested.py") == (
    "if a:\n    try:\n        pass\n    except E +:\n        pass\n")
# An `Error` with no text, a missing newline, prints nothing; the printer's layout puts the next on a line of its own
assert reprinted("return return", "invalid_newline.py") == "return\nreturn\n"
print("ok")